<a href="https://colab.research.google.com/github/omora14/googCol/blob/main/notebooks/starter_bikes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from google.colab import files

train_url = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv'
df_train = pd.read_csv(train_url)

december_url = 'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes_december.csv'
df_december = pd.read_csv(december_url)

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [2]:
def preprocess_unified(df, is_train_data=True, scaler=None):
    """
    A strictly controlled pipeline to guarantee train and holdout data match perfectly.
    """
    data = df.copy()
    data['dteday'] = pd.to_datetime(data['dteday'])

    # 1. Feature Engineering (Class discussions applied here)
    data['month'] = data['dteday'].dt.month
    data['day_of_week'] = data['dteday'].dt.dayofweek

    # Time is a cycle (11PM is next to 12AM)
    data['hr_sin'] = np.sin(2 * np.pi * data['hr'] / 24)
    data['hr_cos'] = np.cos(2 * np.pi * data['hr'] / 24)
    data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)

    # Human Behavior Flags
    data['is_rush_hour'] = data['hr'].apply(lambda x: 1 if x in [7, 8, 9, 16, 17, 18, 19] else 0)
    data['is_weekend'] = data['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

    # Bad Weather Flag (weathersit 3 is snow/heavy rain, 4 is severe)
    data['is_bad_weather'] = data['weathersit'].apply(lambda x: 1 if x >= 3 else 0)

    # 2. Target Variable Logic
    if is_train_data:
        data['total_rentals'] = data['casual'] + data['registered']
        y = data['total_rentals']
    else:
        y = None

    # 3. STRICT COLUMN ENFORCEMENT (This prevents the negative R2 bug)
    feature_cols = [
        'temp_c', 'feels_like_c', 'hum', 'windspeed', 'weathersit', 'season',
        'holiday', 'workingday', 'hr_sin', 'hr_cos', 'month_sin', 'month_cos',
        'is_rush_hour', 'is_weekend', 'is_bad_weather'
    ]

    X = data[feature_cols].copy()

    # 4. Scaling
    scale_cols = ['temp_c', 'feels_like_c', 'hum', 'windspeed']
    if is_train_data:
        scaler = MinMaxScaler()
        X[scale_cols] = scaler.fit_transform(X[scale_cols])
    else:
        if scaler is None:
            raise ValueError("Holdout data requires the fitted training scaler!")
        X[scale_cols] = scaler.transform(X[scale_cols])

    return X, y, scaler

print("Preprocessing engine ready!")

Preprocessing engine ready!


In [3]:
# 1. Process data through our unified pipeline
X_train_full, y_train_full, fitted_scaler = preprocess_unified(df_train, is_train_data=True)

# 2. Split for internal validation
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

# 3. Build the Neural Network Architecture
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1) # Single output for regression
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Stop early if it stops improving to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

print("Training Neural Network... This will take a moment.")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    verbose=1 # Change to 0 if you want to hide the epoch loading bars
)

# --- VERIFICATION STEP ---
val_predictions = model.predict(X_val)
val_predictions = np.maximum(val_predictions, 0) # No negative bikes
r2 = r2_score(y_val, val_predictions)

print("\n" + "="*40)
print(f"INTERNAL VALIDATION R-SQUARED: {r2:.4f}")
print("="*40)
if r2 > 0.75:
    print("✅ Looking excellent! The model is healthy and ready for December data.")
else:
    print("⚠️ R2 is a bit low, but positive. Data is aligned.")

Training Neural Network... This will take a moment.
Epoch 1/100
703/703 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 43991.6523 - mae: 131.3300 - val_loss: 31888.5527 - val_mae: 112.5868 - learning_rate: 0.0010
Epoch 2/100
703/703 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 30309.6348 - mae: 111.4590 - val_loss: 28027.8867 - val_mae: 104.6747 - learning_rate: 0.0010
Epoch 3/100
703/703 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 28782.4258 - mae: 108.8373 - val_loss: 26224.5918 - val_mae: 102.3822 - learning_rate: 0.0010
Epoch 4/100
703/703 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - loss: 27828.2461 - mae: 106.8926 - val_loss: 25460.5352 - val_mae: 100.6643 - learning_rate: 0.0010
Epoch 5/100
703/703 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 27499.4727 - mae: 106.2175 - val_loss: 24937.4707 - val_mae: 99.3170 - learning_rate: 0.0010
Epoch 6/100
703/703 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - loss: 27262.7793 - mae: 105.7112 - val_loss: 25186.9570 - val_mae: 100.1115 - learning_rate: 0.0010
Epoch 7/100


In [9]:
# 1. Process the December holdout using the EXACT same rules and scaler
X_december, _, _ = preprocess_unified(df_december, is_train_data=False, scaler=fitted_scaler)

# 2. Make predictions
december_preds = model.predict(X_december).flatten()

# 3. Format predictions to real-world logic
december_preds = np.maximum(december_preds, 0) # Clip negative numbers to 0
december_preds = np.round(december_preds).astype(int) # Round to whole bikes

# 4. Create the final submission file (Single column named 'predictions')
submission_df = pd.DataFrame({'predictions': december_preds})

# 5. Save and Download
final_filename = 'team4_december_predictions.csv'
submission_df.to_csv(final_filename, index=False)

print(f"\nSuccessfully generated {final_filename} with {len(submission_df)} predictions!")
display(submission_df.head(10))

# Automatically download the file to your computer
files.download(final_filename)

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

Successfully generated team4_december_predictions.csv with 1465 predictions!


,predictions
0,39
1,11
2,3
3,0
4,7
5,50
6,192
7,500
8,816
9,395


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>